# Flow Matching for Image Generation — Colab Setup

Run this notebook to set up the environment, clone the repo, and connect to GitHub.

**GPU check:** Go to Runtime > Change runtime type > Select GPU (T4 is fine).

## 1. Check GPU availability

In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_total_memory(0) / 1e9:.1f} GB")

## 2. Mount Google Drive (for saving checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create a folder for this project on Drive
import os
DRIVE_DIR = '/content/drive/MyDrive/flow-matching'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Drive folder ready: {DRIVE_DIR}")

## 3. Clone the repo and install dependencies

In [ ]:
%cd /content
!git clone https://github.com/alidor4702/Flow-Matching-Image-Generation.git
%cd Flow-Matching-Image-Generation
!pip install -q -r requirements.txt

## 4. Set up GitHub authentication

**Never hardcode your token.** This uses `getpass` so it stays hidden.

In [ ]:
from getpass import getpass

token = getpass("Enter your GitHub personal access token: ")
!git remote set-url origin https://{token}@github.com/alidor4702/Flow-Matching-Image-Generation.git

# Configure git identity
!git config user.name "alidor4702"
!git config user.email "your-email@example.com"  # <-- change this

print("GitHub auth configured.")

## 5. Switch to your branch

In [ ]:
# Create or switch to your development branch
BRANCH = "ali/dev"  # change to "elora/dev" for Elora

!git checkout -b {BRANCH} || git checkout {BRANCH}
!git pull origin {BRANCH} || echo "New branch, nothing to pull."

## 6. Helper: Save checkpoints to Drive

In [ ]:
import shutil

def save_to_drive(src_path, tag="latest"):
    """Copy a checkpoint file to Google Drive for persistence."""
    dst = os.path.join(DRIVE_DIR, f"checkpoint_{tag}.pt")
    shutil.copy2(src_path, dst)
    print(f"Saved to {dst}")

def load_from_drive(tag="latest"):
    """Load a checkpoint path from Google Drive."""
    path = os.path.join(DRIVE_DIR, f"checkpoint_{tag}.pt")
    if os.path.exists(path):
        print(f"Found checkpoint: {path}")
        return path
    print("No checkpoint found.")
    return None

## 7. Helper: Push results to GitHub

In [ ]:
def push_results(message="Update results"):
    """Stage results and push to your branch."""
    !git add results/ notebooks/
    !git commit -m "{message}"
    !git push origin {BRANCH}
    print("Pushed to GitHub.")

---

## Ready to go!

You can now start training. Example:

```python
# Your training code here
# ...

# After training, save checkpoint
save_to_drive("results/checkpoints/model.pt", tag="epoch_100")

# Push results to GitHub
push_results("CFM training run — epoch 100")
```